In [1]:
# Load libraries
import pyhocon
from tqdm import tqdm
from itertools import combinations
from sklearn.utils import shuffle
from torch.utils import data

from models import  SpanEmbedder, SpanScorer, FullCrossEncoder
from evaluator import Evaluation
from dataset import CrossEncoderDatasetFull,CrossEncoderMixedDatasetFull, ParaphraseDataset
from utils import *


/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/transformers/generation_utils.py:24: FutureWarning: Importing `GenerationMixin` from `src/transformers/generation_utils.py` is deprecated and will be removed in Transformers v5. Import as `from transformers import GenerationMixin` instead.
  warnings.warn(
Xformers is not installed correctly. If you want to use memorry_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [2]:
#Load config files
config = pyhocon.ConfigFactory.parse_file('configs/config_pairwise.json')
fix_seed(config)
logger = create_logger(config, create_file=True)
logger.info(pyhocon.HOCONConverter.convert(config, "hocon"))
create_folder(config.model_path)
print(config.experiment_id)

None


In [3]:
config_experiments = pyhocon.ConfigFactory.parse_file('configs/config_experiments.json')
datasets_config = pyhocon.ConfigFactory.parse_file('configs/config_datasets.json')
datasets_config = resolve_data_path(datasets_config)

### Load datasets according to config file

In [4]:
config.data_folder

'../../CDCR/data/ECBplus-prep/output_data/'

In [5]:
# init train and dev set
if "Experiment" in config.experiment_id:
    train = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['train'])
    val = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['val'])
    #test = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['test'])

else:
    if config.data_folder == "mrpc" or "HoW_DatasetDict" in config.data_folder:
        train = ParaphraseDataset(config, 'train') 
        val = ParaphraseDataset(config, 'validation')
    else:
        train = CrossEncoderDatasetFull(config, 'train') 
        val = CrossEncoderDatasetFull(config, 'val')

/home/sago/Documents/GitHub/CDCR_lexical_diversity/MTL/cross_encoder_adapted/dataset.py:145: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels,dtype=torch.float)


### Debug problematic mentions

In [6]:
# topics_list = set([mention["topic_id"] for mention in val.mentions])
# subtopics_list = set([mention["subtopic_id"] for mention in val.mentions])

# problematic_mentions = [mention for mention in train.mentions if min(mention['tokens_number_context'])>len(mention["mention_context"])]

# ids= []
# len_context = []
# min_mention_token_number = []
# for mention in problematic_mentions:
#     ids.append(mention["mention_id"])
#     len_context.append(len(mention["mention_context"]))
#     min_mention_token_number.append(min(mention['tokens_number_context']))

# import pandas as pd
# df = pd.DataFrame({"mention_id":ids,"context_length":len_context,"min_mention_token_number":min_mention_token_number})
# # Print the DataFrame in Markdown-friendly format
# markdown_table = df.to_markdown(index=False)
# print(len(df))
# print(markdown_table)

## Create dataloaders, device and model

In [7]:
#Subset data in debug for dummy run
train_subset = data.Subset(train, range(8))
val_subset = data.Subset(val, range(8))

In [14]:
train_loader = data.DataLoader(train_subset, batch_size=config.batch_size, shuffle=True)
val_loader = data.DataLoader(val_subset, batch_size=config.batch_size, shuffle=False)

#device_ids = config.gpu_num
#device = torch.device("cuda:{}".format(device_ids[0]))
device = torch.device("cpu")

logger.info('Init models')
cross_encoder_single = FullCrossEncoder(config).to(device)
#cross_encoder = torch.nn.DataParallel(cross_encoder_single, device_ids=[device])
cross_encoder = cross_encoder_single

loading file vocab.json from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/vocab.json
loading file merges.txt from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/merges.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at None
loading configuration file config.json from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/config.json
Model config RobertaConfig {
  "_name_or_path": "roberta-large",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 

In [15]:
## Optimizer and loss function
criterion = get_loss_function(config)
optimizer = get_optimizer(config, [cross_encoder])
scheduler = get_scheduler(optimizer, total_steps=config.epochs * len(train_loader))



In [16]:
model_params = list(cross_encoder.parameters())

# with open('model_parameters.txt', 'w') as f:
#     for name, param in cross_encoder.named_parameters():
#         f.write(f"Layer: {name}, Parameters: {param.numel()}\n")

for name, param in cross_encoder.named_parameters():
    print(f"Layer: {name}, Parameters: {param.numel()}")

Layer: model.embeddings.word_embeddings.weight, Parameters: 51473408
Layer: model.embeddings.position_embeddings.weight, Parameters: 526336
Layer: model.embeddings.token_type_embeddings.weight, Parameters: 1024
Layer: model.embeddings.LayerNorm.weight, Parameters: 1024
Layer: model.embeddings.LayerNorm.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.self.query.weight, Parameters: 1048576
Layer: model.encoder.layer.0.attention.self.query.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.self.key.weight, Parameters: 1048576
Layer: model.encoder.layer.0.attention.self.key.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.self.value.weight, Parameters: 1048576
Layer: model.encoder.layer.0.attention.self.value.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.output.dense.weight, Parameters: 1048576
Layer: model.encoder.layer.0.attention.output.dense.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.output.LayerNorm.weight, Param

In [17]:
# Revision prior to training
cross_encoder.train()
# Get one batch
batch_x, batch_y, instance_id, dataset_task = next(iter(train_loader))

before_sep = [parts[0] for parts in (string.split("[SEP]") for string in batch_x)]
after_sep = [parts[1] for parts in (string.split("[SEP]") for string in batch_x)]

bert_tokens = cross_encoder_single.tokenizer(text=before_sep,text_pair=after_sep,add_special_tokens=True,padding=True,truncation=True)
cross_encoder_single.tokenizer.decode(bert_tokens['input_ids'][0])

"<s>Perennial party girl Tara Reid checked herself into Promises Treatment Center, her rep told People. `` We appreciate your respect to her and her family's privacy at this time, '' the 33-year-old actress's rep Jack Ketsoyan told the magazine exclusively for their Web site. Ketsoyan would not say what the `` American Pie '' star was being treated for. A friend of the actress told People she went to Promises [START] on Tuesday [END] and that her friends and family supported her decision. Reid launched a new fashion line called Mantra in December. </s></s> Tara Reid has entered a rehab program, the American Pie star's rep confirmed to People and Us Weekly [START] on Friday [END]. The news comes as a shock for everyone who just assumed the seldom-seen actress had been holed up there for a while - at least when she wasn't touring U.S. frat houses and/or tequila conventions. `` Tara Reid has checked herself into Promises Treatment Center, '' Reid's rep told Us. `` We appreciate your respe

In [18]:
##################################################################################
####                    TRAINING
##################################################################################

f1 = []
best_f1 = 0.0
update_interval = 10

for epoch in range(config.epochs):
    logger.info('Epoch: {}'.format(epoch))
    accumulate_loss = 0
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.train()

    for batch_x, batch_y, instance_id, dataset_task in tqdm(train_loader):
        #Separate sentences from batch to add the tokenizers special tokens
        before_sep = [parts[0] for parts in (string.split("[SEP]") for string in batch_x)]
        after_sep = [parts[1] for parts in (string.split("[SEP]") for string in batch_x)]

        bert_tokens = cross_encoder_single.tokenizer(text=before_sep,text_pair=after_sep,add_special_tokens=True,padding=True,truncation=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)

        optimizer.zero_grad()
        cross_encoder.zero_grad()
        scores = cross_encoder(input_ids, attention_mask,dataset_task)

        batch_y = batch_y.view(-1, 1)
        loss = criterion(scores, batch_y.to(device))
        loss.mean().backward()
        torch.nn.utils.clip_grad_norm_(cross_encoder.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        torch.cuda.empty_cache()


        accumulate_loss += loss.item()

        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)

    # list_of_topics = shuffle(list(range(len(training_set.topic_list))))
    # number_of_positive_pairs = 0
    # number_of_pairs = 0
    # for topic_num in tqdm(list_of_topics):
    #
    #
    #
    #
    #     topic = training_set.topic_list[topic_num]
    #     first, second, labels = get_candidate_spans(training_set, topic)
    #
    #     number_of_positive_pairs += len((labels == 1).nonzero())
    #     number_of_pairs += len(labels)
    #
    #
    #     pairwises = CrossEncoderDataset(training_set.mentions_repr, first, second, labels)
    #     generator = data.DataLoader(pairwises, shuffle=True, batch_size=config.batch_size)
    #     loss = fine_tune_bert(cross_encoder, generator, criterion, optimizer, device)
    #     accumulate_loss += loss


    logger.info('Number of positive/total pairs: {}/{}'.format(number_of_positive_pairs, number_of_pairs))
    logger.info('Accumulate loss: {}'.format(accumulate_loss))


    logger.info('Evaluate on the dev set')
    all_scores, all_labels = [], []
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.eval()

    for batch_x, batch_y,instance_id,dataset_task in tqdm(val_loader):
        bert_tokens = cross_encoder_single.tokenizer(batch_x, pad_to_max_length=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)

        with torch.no_grad():
            scores = cross_encoder(input_ids, attention_mask,dataset_task)

        torch.cuda.empty_cache()
        
        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)


    # for topic_num, topic in enumerate(tqdm(dev_set.topic_list)):
    #     dev_topic = dev_set.topic_list[topic_num]
    #     first, second, labels = get_candidate_spans(dev_set, dev_topic)
    #     number_of_positive_pairs += len((labels == 1).nonzero())
    #     number_of_pairs += len(labels)
    #
    #     pairwises = CrossEncoderDataset(dev_set.mentions_repr, first, second, labels)
    #     generator = data.DataLoader(pairwises, shuffle=True, batch_size=config.batch_size)
    #     scores, labels = evaluate_model(cross_encoder, generator, device)
    #     torch.cuda.empty_cache()
    #
        all_scores.extend(scores.squeeze(1))
        all_labels.extend(batch_y.squeeze(1))



    all_labels = torch.stack(all_labels)
    all_scores = torch.stack(all_scores)


    strict_preds = (all_scores > 0).to(torch.int)
    eval = Evaluation(strict_preds, all_labels.to(device))
    logger.info('Number of predictions: {}/{}'.format(strict_preds.sum(), len(strict_preds)))
    logger.info('Number of positive pairs: {}/{}'.format(len((all_labels == 1).nonzero()),
                                                            len(all_labels)))

    logger.info('Min score: {}'.format(all_scores.min().item()))
    logger.info('Max score: {}'.format(all_scores.max().item()))
    logger.info('Strict - Recall: {}, Precision: {}, F1: {}, Accuracy: {}'.
                format(eval.get_recall(), eval.get_precision(), eval.get_f1(), eval.get_accuracy()))


    out_dir = os.path.join(config.model_path, '{}_{}'.format(
        'base' if 'base' in config.roberta_model else 'large', config.batch_size),
        'checkpoint_{}'.format(epoch))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    current_f1 = eval.get_f1() 
    if current_f1 > best_f1:
        best_f1 = current_f1
        cross_encoder_single.model.save_pretrained(os.path.join(out_dir, 'bert'))
        torch.save(cross_encoder_single.linear_cdcr.state_dict(), os.path.join(out_dir, 'linear_cdcr'))
        torch.save(cross_encoder_single.linear_cdcr.state_dict(), os.path.join(out_dir, 'linear_pi'))

    f1.append(eval.get_f1())
print("F1 values",f1)


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:2364: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
100%|██████████| 2/2 [00:05<00:00,  2.74s/it]
Configuration saved in models/pairwise_scorers/large_4/checkpoint_0/bert/config.json
Model weights saved in models/pairwise_scorers/large_4/checkpoint_0/bert/pytorch_model.bin
 50%|█████     | 1/2 [00:12<00:12, 12.06s/it]

: 

: 

In [ ]:
f1